# Analyse exploratoire : espérance de vie (OMS)

Projet Dazzle Dash. Dataset : *Life Expectancy Data* de l'OMS, 193 pays, 2000-2015.

Ce notebook me sert à comprendre les données avant de construire le dashboard. Je ne cherche pas encore à faire une application : je veux savoir ce que contient le fichier, ce qui manque, ce qui est faux, et quels liens ressortent entre les variables.

Je pars de la question du brief : quels facteurs sont associés à l'espérance de vie, et comment varient-ils selon les pays ? Pour y répondre, je me pose quatre questions plus précises.

1. Quel est le niveau moyen de l'espérance de vie et comment évolue-t-il ?
2. Quelles inégalités existe-t-il entre pays et entre pays développés et en développement ?
3. Quelles variables sont les plus liées à l'espérance de vie ?
4. Quelles limites m'empêchent de conclure à une causalité ?

## Comment j'ai procédé

J'ai suivi l'ordre habituel d'une analyse exploratoire :

- comprendre la structure du tableau ;
- regarder les valeurs manquantes et leur mécanisme ;
- vérifier les valeurs aberrantes une par une avant de les traiter ;
- nettoyer en justifiant chaque décision ;
- décrire les variables, puis les croiser ;
- regarder l'évolution dans le temps ;
- terminer par un bilan et des chiffres réutilisables dans le dashboard.

Je note aussi les limites au fur et à mesure. C'est important pour ne pas faire dire aux données plus qu'elles ne disent.

In [1]:
# Bibliothèques, chemins et paramètres communs à tout le notebook.
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import pycountry
from plotly.subplots import make_subplots

# Je garde les figures interactives dans le notebook.
# Le format "plotly_mimetype" évite d'embarquer tout Plotly.js dans chaque sortie.
pio.renderers.default = "plotly_mimetype"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Le notebook peut être lancé depuis la racine du projet ou depuis le dossier notebooks.
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = RACINE / "data" / "Life_Expectancy_Data.csv"
DOSSIER_CLEAN = RACINE / "data" / "clean"
DATA_OBS = DOSSIER_CLEAN / "life_expectancy_observations.csv"
DATA_DASH = DOSSIER_CLEAN / "life_expectancy_dashboard.csv"
DOSSIER_CLEAN.mkdir(parents=True, exist_ok=True)

# Même palette que celle que je réutiliserai dans Dash.
BLEU = "#1f4e79"
ORANGE = "#e07b39"
VERT = "#2e8b57"
GRIS = "#6b7280"

STATUTS = {"Developed": "Développé", "Developing": "En développement"}
COULEURS_STATUT = {"Développé": BLEU, "En développement": ORANGE}

# Le fichier d'origine contient des espaces dans les noms de colonnes.
# Je les retire avant de renommer, sinon les clés ne correspondent pas.
RENOMMAGE = {
    "Country": "country",
    "Year": "year",
    "Status": "status",
    "Life expectancy": "life_expectancy",
    "Adult Mortality": "adult_mortality",
    "infant deaths": "infant_deaths",
    "Alcohol": "alcohol",
    "percentage expenditure": "health_exp",
    "Hepatitis B": "hepatitis_b",
    "Measles": "measles",
    "BMI": "bmi",
    "under-five deaths": "under_five_deaths",
    "Polio": "polio",
    "Total expenditure": "total_exp",
    "Diphtheria": "diphtheria",
    "HIV/AIDS": "hiv_aids",
    "GDP": "gdp",
    "Population": "population",
    "thinness  1-19 years": "thinness_10_19",
    "thinness 5-9 years": "thinness_5_9",
    "Income composition of resources": "income_comp",
    "Schooling": "schooling",
}

# Les libellés français servent pour les tableaux et les graphiques.
LABELS = {
    "country": "Pays",
    "year": "Année",
    "status": "Statut",
    "statut": "Statut",
    "life_expectancy": "Espérance de vie (années)",
    "adult_mortality": "Mortalité adulte (pour 1 000)",
    "infant_deaths": "Décès infantiles (pour 1 000)",
    "under_five_deaths": "Décès avant 5 ans (pour 1 000)",
    "alcohol": "Consommation d'alcool (L/hab.)",
    "health_exp": "Dépenses de santé par habitant (USD, proxy)",
    "health_exp_log": "Dépenses de santé (log10 USD/hab., 2000-2014)",
    "health_exp_zero": "Dépenses de santé nulles",
    "health_exp_disponible": "Dépense de santé exploitable",
    "gdp_aberrant": "PIB signalé comme aberrant",
    "gdp_clean": "PIB par habitant nettoyé (USD)",
    "hepatitis_b": "Vaccination hépatite B (%)",
    "measles": "Cas de rougeole (pour 1 000)",
    "bmi": "IMC moyen",
    "bmi_suspect": "IMC signalé comme suspect",
    "polio": "Vaccination polio (%)",
    "total_exp": "Dépenses publiques de santé (% du budget)",
    "diphtheria": "Vaccination DTP3 (%)",
    "hiv_aids": "Décès VIH/sida (pour 1 000 naissances)",
    "gdp": "PIB par habitant (USD)",
    "population": "Population",
    "thinness_10_19": "Maigreur 10-19 ans (%)",
    "thinness_5_9": "Maigreur 5-9 ans (%)",
    "income_comp": "Indice de composition des revenus (0-1)",
    "schooling": "Scolarité (années)",
}


def marquer_pib_aberrant(frame):
    # Je signale les PIB qui rompent trop fortement la série du pays.
    # Je ne modifie pas gdp : je crée gdp_aberrant et gdp_clean.
    tri = frame.sort_values(["country", "year"]).copy()
    mediane_pays = tri.groupby("country")["gdp"].transform("median")
    variation = tri.groupby("country")["gdp"].pct_change()
    tri["gdp_aberrant"] = (
        tri["gdp"].lt(100)
        | tri["gdp"].lt(0.1 * mediane_pays)
        | tri["gdp"].gt(10 * mediane_pays)
        | variation.lt(-0.9)
        | variation.gt(9)
    )
    tri["gdp_clean"] = tri["gdp"].where(~tri["gdp_aberrant"])
    return tri.sort_index()


def correlation_partielle(x, y, z):
    # Corrélation de x et y en contrôlant z, sur les lignes complètes.
    masque = x.notna() & y.notna() & z.notna()
    x, y, z = x[masque], y[masque], z[masque]
    if len(x) < 3:
        return np.nan, len(x)
    rxy = x.corr(y)
    rxz = x.corr(z)
    ryz = y.corr(z)
    denominateur = np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2))
    if denominateur == 0:
        return np.nan, len(x)
    return (rxy - rxz * ryz) / denominateur, len(x)

## 1. Charger le fichier

Je commence par vérifier les dimensions, la période et ce que représente une ligne. C'est ce qui me dira si je peux comparer les pays entre eux et suivre leur évolution.

In [2]:
# Je charge le fichier brut et je vérifie sa structure avant de toucher aux données.
df_brut = pd.read_csv(DATA_RAW)

# Nettoyage minimal des noms de colonnes avant le renommage.
df_brut.columns = df_brut.columns.str.strip()
df = df_brut.rename(columns=RENOMMAGE)

print(f"Lignes : {df.shape[0]:,}")
print(f"Colonnes : {df.shape[1]}")
print(f"Pays : {df['country'].nunique()}")
print(f"Période : {df['year'].min()} - {df['year'].max()}")
print("\nNombre d'années disponibles par pays :")
print(df.groupby("country").size().value_counts().sort_index().to_string())

df.head()

Lignes : 2,938
Colonnes : 22
Pays : 193
Période : 2000 - 2015

Nombre d'années disponibles par pays :
1      10
16    183


,country,year,status,life_expectancy,adult_mortality,infant_deaths,alcohol,health_exp,hepatitis_b,measles,bmi,under_five_deaths,polio,total_exp,diphtheria,hiv_aids,gdp,population,thinness_10_19,thinness_5_9,income_comp,schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,19.1,83,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,18.6,86,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,18.1,89,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,17.6,93,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,17.2,97,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


Le fichier contient 2 938 lignes et 22 colonnes. Une ligne correspond à un pays pour une année.

La structure n'est pas tout à fait équilibrée : 183 pays sont présents sur les 16 années, soit 2 928 lignes. Les 10 autres pays n'apparaissent qu'en 2013 et n'ont pas d'espérance de vie renseignée. Cela explique la différence entre 193 pays et 183 pays réellement exploitables.

Je retiens donc deux choses pour la suite : je travaille sur un panel pays-année, et je devrai supprimer les 10 lignes sans espérance de vie pour les analyses qui portent sur cette variable.

## 2. Regarder ce qui manque

Avant de calculer quoi que ce soit, je veux savoir quelles colonnes sont incomplètes et si les valeurs manquantes sont réparties au hasard. C'est ce point qui détermine la façon dont je peux les traiter.

In [3]:
# Je trie les variables par taux de manquants pour repérer les colonnes à problème.
variables_qualite = [
    "life_expectancy", "adult_mortality", "alcohol", "health_exp", "hepatitis_b",
    "bmi", "polio", "total_exp", "diphtheria", "hiv_aids", "gdp", "population",
    "thinness_10_19", "thinness_5_9", "income_comp", "schooling",
]

bilan_manquants = pd.DataFrame({
    "Variable": [LABELS[c] for c in variables_qualite],
    "Manquants": [df[c].isna().sum() for c in variables_qualite],
    "Zéros": [int((df[c] == 0).sum()) for c in variables_qualite],
    "Part (%)": [round(df[c].isna().mean() * 100, 1) for c in variables_qualite],
})
bilan_manquants.sort_values("Part (%)", ascending=False)

,Variable,Manquants,Zéros,Part (%)
11,Population,652,0,22.2
4,Vaccination hépatite B (%),553,0,18.8
10,PIB par habitant (USD),448,0,15.2
7,Dépenses publiques de santé (% du budget),226,0,7.7
2,Consommation d'alcool (L/hab.),194,0,6.6
14,Indice de composition des revenus (0-1),167,130,5.7
15,Scolarité (années),163,28,5.5
5,IMC moyen,34,0,1.2
13,Maigreur 5-9 ans (%),34,0,1.2
12,Maigreur 10-19 ans (%),34,0,1.2


In [4]:
# Je retire uniquement les 10 lignes sans espérance de vie :
# elles ne servent à rien pour expliquer cette variable.
df_analyse = df.dropna(subset=["life_expectancy"]).copy()

# Les zéros de health_exp sont des données manquantes déguisées :
# je les compte comme manquants pour lire le graphique.
df_manquants = df_analyse.copy()
df_manquants["health_exp"] = df_manquants["health_exp"].replace(0, np.nan)

variables_manquantes = [
    "gdp", "population", "hepatitis_b", "health_exp", "total_exp", "alcohol",
    "income_comp", "schooling", "bmi", "polio", "diphtheria",
]

# Taux de valeurs manquantes par année et par variable.
taux = (
    df_manquants.groupby("year")[variables_manquantes]
    .apply(lambda groupe: groupe.isna().mean() * 100)
    .T
)
taux.index = [LABELS.get(c, c) for c in taux.index]

fig = px.imshow(
    taux,
    color_continuous_scale="Reds",
    zmin=0,
    zmax=100,
    labels={"x": "Année", "y": "", "color": "% manquant"},
    title="Part de valeurs manquantes par année",
    aspect="auto",
)
fig.update_layout(template="plotly_white", height=560, coloraxis_colorbar_title="%")
fig.show()

Trois choses ressortent de ce graphique.

La vaccination contre l'hépatite B est très incomplète au début de la période : environ 54 % de valeurs manquantes en 2000, contre moins de 5 % en 2015. Ce n'est donc pas un manque aléatoire, c'est un problème de remontée d'information qui s'améliore avec le temps.

Les dépenses publiques de santé et la consommation d'alcool sont presque absentes en 2015 : respectivement 98,9 % et 96,7 % de valeurs manquantes. Ces deux variables ne sont donc pas exploitables pour l'année 2015, même si elles le sont plutôt bien avant.

Le PIB et la population manquent de façon plus stable, autour de 15 % et 22 %. Pour les dépenses de santé, il n'y a presque pas de NaN, mais les zéros jouent le même rôle : 606 zéros après retrait des 10 lignes sans espérance de vie, dont 181 en 2015. En 2015, seules 2 valeurs sont non nulles.

In [5]:
# Pour savoir si le manque est lié à l'espérance de vie,
# je compare la moyenne des pays renseignés et des pays non renseignés.
variables_test = ["gdp", "hepatitis_b", "schooling", "total_exp", "alcohol"]

lignes = []
for variable in variables_test:
    for etat, masque in [
        ("Donnée présente", df_analyse[variable].notna()),
        ("Donnée manquante", df_analyse[variable].isna()),
    ]:
        lignes.append({
            "Variable": LABELS[variable],
            "État": etat,
            "Espérance de vie moyenne": df_analyse.loc[masque, "life_expectancy"].mean(),
        })

comparaison = pd.DataFrame(lignes)

fig = px.bar(
    comparaison,
    x="Variable",
    y="Espérance de vie moyenne",
    color="État",
    barmode="group",
    color_discrete_map={"Donnée présente": BLEU, "Donnée manquante": ORANGE},
    labels={"Variable": "", "Espérance de vie moyenne": "Espérance de vie moyenne (années)"},
    title="Quand une donnée manque, l'espérance de vie est-elle différente ?",
    text_auto=True,
)
fig.update_layout(template="plotly_white", height=480, xaxis_tickangle=-20)
fig.show()

Le manque n'est pas anodin partout.

Pour l'hépatite B, les pays sans données ont une espérance de vie moyenne de 66,1 ans, contre 69,9 ans pour les pays renseignés. L'écart est important. Si j'imputais simplement la moyenne mondiale, je donnerais donc aux pays mal renseignés une valeur qui ne correspond probablement pas à leur situation. Pour la scolarité, l'écart est de 67,1 contre 69,3 ans. Pour le PIB, il est plus faible : 68,4 contre 69,4 ans. Pour les dépenses publiques de santé et l'alcool, il est quasi nul, mais ces variables sont surtout absentes en 2015.

Je ne peux pas prouver le mécanisme à partir de ces seules données, mais je peux au moins exclure que tout soit « manquant complètement au hasard ».

### Ce que ça change dans ma façon de traiter les manquants

En cours, on m'a présenté quatre grandes familles de stratégies, de la plus simple à la plus fidèle. Je les reprends ici parce qu'elles correspondent bien à ce que j'observe.

| Stratégie | Principe | Quand c'est raisonnable |
|---|---|---|
| Supprimer | On enlève les lignes ou les colonnes incomplètes | Peu de manquants et manque complètement aléatoire (MCAR) |
| Imputer simplement | Moyenne, médiane, mode, constante, interpolation | Exploration rapide ou base de comparaison |
| Imputer par modèle | KNN, MICE, forêts aléatoires | Variables liées entre elles, manque de type MAR |
| Garder l'information | Indicateur de manquant, modèles qui acceptent les NaN | Le fait de manquer est lui-même porteur de sens |

Quelques définitions utiles :

- **MCAR** : le manque ne dépend ni des variables observées ni de la valeur manquante. Supprimer les lignes est alors peu risqué.
- **MAR** : le manque dépend de variables observées, par exemple l'année ou le statut du pays. Une imputation par groupe ou par modèle peut être envisagée.
- **MNAR** : le manque dépend de la valeur elle-même, par exemple un pays ne déclare pas sa vaccination parce que son système de santé est faible. Dans ce cas, aucun modèle ne corrige complètement le biais.

Je ne vais pas appliquer une seule stratégie à tout le fichier. Je choisis par variable, selon le taux de manquants, le mécanisme probable et l'usage que je veux en faire.

### Décisions par variable

| Variable | Manquants | Ce que j'observe | Ce que je décide |
|---|---:|---|---|
| Espérance de vie | 10 lignes (0,3 %) | uniquement les 10 petits États présents en 2013 | supprimer ces lignes |
| Population | 22,0 % | 0/143 séries cohérentes, 703 transitions > ×10, 129 > ×100 | exclure de l'analyse |
| Vaccination hépatite B | 18,9 % | manque surtout avant 2005 ; les pays non renseignés ont une espérance de vie plus faible | garder uniquement les valeurs observées, pas d'imputation pour les corrélations |
| PIB par habitant | 15,1 % | 25 pays sans aucun PIB dans le fichier nettoyé ; 622 observations signalées comme aberrantes | garder `gdp` brut pour la traçabilité, utiliser `gdp_clean` pour les tendances |
| Dépenses publiques de santé | 7,7 % | presque aucune donnée en 2015 | limiter les analyses à 2000-2014 |
| Consommation d'alcool | 6,6 % | presque aucune donnée en 2015 | limiter les analyses à 2000-2014 |
| Scolarité et indice de revenu | 5,5 %, puis 6,4 % après traitement des zéros | 26 zéros de scolarité qui ressemblent à des données manquantes déguisées ; les deux variables manquent souvent ensemble | transformer les zéros en valeurs manquantes, puis interpoler dans le temps pour l'affichage |
| Vaccination polio et DTP3 | 0,6 % | manquent ensemble, très peu de cas | interpolation simple pour l'affichage |
| IMC | 1,1 % + 282 valeurs aberrantes | séries nationales incohérentes et échelle douteuse | exclure de l'analyse |
| Dépenses de santé par habitant | 0 NaN + 611 zéros (606 après retrait des 10 lignes) | 1 301 valeurs > 100, maximum 19 480 ; 31 pays n'ont que des zéros ; 2015 presque vide | traiter les zéros comme manquants, passer en log10, limiter à 2000-2014, garder comme variable secondaire |

Cette fois, je ne me contente pas de remplacer les valeurs gênantes par des NaN. Je vérifie d'abord si la colonne elle-même est utilisable.

## 3. Vérifier les valeurs aberrantes

Une valeur extrême n'est pas forcément une erreur. Une épidémie de rougeole peut produire un nombre de cas très élevé, et un pays peut réellement avoir une mortalité adulte exceptionnelle. Je distingue donc trois cas : impossible, suspect, et réel mais extrême.

In [6]:
# Je regarde les séries nationales autour des valeurs faibles d'IMC.
# Si une valeur basse est isolée entre deux valeurs normales, c'est probablement une erreur de saisie.
exemples_bmi = ["Albania", "Australia", "Viet Nam", "Luxembourg"]

bmi_series = (
    df_analyse[df_analyse["country"].isin(exemples_bmi)]
    .pivot(index="year", columns="country", values="bmi")
    .round(1)
)
print(f"Nombre d'IMC inférieurs à 10 : {(df_analyse['bmi'] < 10).sum()}")
print(f"Nombre d'IMC supérieurs à 80 : {(df_analyse['bmi'] > 80).sum()}")
bmi_series

Nombre d'IMC inférieurs à 10 : 282
Nombre d'IMC supérieurs à 80 : 0


country,Albania,Australia,Luxembourg,Viet Nam
year,,,,
2000,45.0,58.2,54.0,9.2
2001,46.0,59.0,54.6,9.6
2002,46.9,59.6,55.2,1.0
2003,47.9,6.3,55.8,1.4
2004,48.9,6.9,56.4,1.9
2005,49.9,61.5,57.0,11.3
2006,5.8,62.0,57.5,11.8
2007,51.7,62.5,57.9,12.3
2008,52.6,62.9,58.4,12.9


Les valeurs basses sont isolées. Par exemple, l'Albanie passe de 49,9 en 2005 à 5,8 en 2006, puis à 51,7 en 2007. L'Australie a 59,6 en 2002, puis 6,3 et 6,9 en 2003 et 2004, avant de revenir à 61,5. Ces valeurs ressemblent à des erreurs de saisie ou à un décalage d'échelle sur quelques lignes.

Mais il y a un problème plus gênant : même les valeurs « normales » de cette colonne sont élevées. L'Albanie tourne autour de 45-58 et l'Australie autour de 58-66, alors que les IMC moyens connus sont plutôt autour de 25-28. Je ne peux pas corriger cela avec une simple règle de valeurs extrêmes. Je décide donc de ne pas utiliser l'IMC dans mes conclusions, même après avoir neutralisé les valeurs inférieures à 10.

In [7]:
# Même vérification pour la population : je regarde quelques séries complètes.
exemples_population = ["China", "Croatia", "Maldives", "Hungary"]

population_series = (
    df_analyse[df_analyse["country"].isin(exemples_population)]
    .pivot(index="year", columns="country", values="population")
)
population_series

country,China,Croatia,Hungary,Maldives
year,,,,
2000,1262645.0,4426.0,121971.0,286.0
2001,127185.0,444.0,1187576.0,292.0
2002,1284.0,444.0,115868.0,297.0
2003,12884.0,444.0,1129552.0,34.0
2004,129675.0,4439.0,117146.0,312.0
2005,13372.0,4442.0,18765.0,321.0
2006,13112.0,444.0,17137.0,333.0
2007,1317885.0,4436.0,15578.0,349.0
2008,1324655.0,443458.0,138188.0,362.0


In [8]:
# Je calcule les variations annuelles de population dans chaque pays.
population_triee = df_analyse.dropna(subset=["population"]).sort_values(["country", "year"]).copy()
population_triee["variation"] = population_triee.groupby("country")["population"].pct_change()

hausses_10 = int((population_triee["variation"] > 9).sum())
baisses_10 = int((population_triee["variation"] < -0.9).sum())
hausses_100 = int((population_triee["variation"] > 99).sum())
baisses_100 = int((population_triee["variation"] < -0.99).sum())

variations_par_pays = population_triee.groupby("country")["variation"].agg(list)
pays_evalues = sum(1 for valeurs in variations_par_pays if any(pd.notna(v) for v in valeurs))
pays_coherents = sum(
    1 for valeurs in variations_par_pays
    if all(-0.5 <= v <= 1.0 for v in valeurs if pd.notna(v))
)

print(f"Transitions > x10 : {hausses_10 + baisses_10} ({hausses_10} hausses, {baisses_10} baisses)")
print(f"Transitions > x100 : {hausses_100 + baisses_100} ({hausses_100} hausses, {baisses_100} baisses)")
print(f"Pays avec au moins deux valeurs : {pays_evalues}")
print(f"Pays avec une série cohérente (variations entre x0,5 et x2) : {pays_coherents}")

sauts = population_triee[population_triee["variation"].abs() > 9]
sauts[["country", "year", "population", "variation"]].head(12)

Transitions > x10 : 703 (493 hausses, 210 baisses)
Transitions > x100 : 129 (91 hausses, 38 baisses)
Pays avec au moins deux valeurs : 143
Pays avec une série cohérente (variations entre x0,5 et x2) : 0


,country,year,population,variation
14,Afghanistan,2001,2966463.0,9.098391
11,Afghanistan,2004,24118979.0,9.198942
9,Afghanistan,2006,2589345.0,9.044085
8,Afghanistan,2007,26616792.0,9.279353
5,Afghanistan,2010,2883167.0,9.140178
0,Afghanistan,2015,33736494.0,101.986410
28,Albania,2003,339616.0,95.729137
23,Albania,2008,2947314.0,98.179392
18,Albania,2013,289592.0,97.467188
46,Algeria,2001,31592153.0,9.130996


La colonne Population n'est pas seulement incomplète : les unités ne sont pas cohérentes. La Chine passe de 1 262 645 à 127 185, puis à 1 284, puis à 12 884, avant de remonter à 129 675. Selon les années, la valeur est exprimée en unités, en milliers ou divisée par 100.

J'ai compté 703 transitions supérieures à un facteur 10 (493 hausses, 210 baisses) et 129 supérieures à un facteur 100 (91 hausses, 38 baisses). Aucun des 143 pays avec au moins deux valeurs n'a une série cohérente : 0/143. Ce n'est donc pas un problème marginal, et garder seulement les pays « cohérents » ne donnerait aucune ligne. La colonne n'est pas exploitable pour classer les pays ou pondérer des moyennes. Je l'exclus complètement.

In [9]:
# La variable "percentage expenditure" est-elle vraiment un pourcentage ?
zeros_sante = df_analyse["health_exp"].eq(0)
ratio_sante_pib = (df_analyse["health_exp"] / df_analyse["gdp"]).replace([np.inf, -np.inf], np.nan)

print(f"Zéros de dépenses de santé : {int(zeros_sante.sum())}")
print(f"Zéros en 2015 : {int((zeros_sante & df_analyse['year'].eq(2015)).sum())}")
print(f"Valeurs non nulles en 2015 : {int((~zeros_sante & df_analyse['year'].eq(2015)).sum())}")
print(f"Valeurs supérieures à 100 : {int((df_analyse['health_exp'] > 100).sum())}")
print(f"Valeurs supérieures à 1 000 : {int((df_analyse['health_exp'] > 1_000).sum())}")
print(f"Ratio médian dépenses de santé / PIB : {ratio_sante_pib.median():.3f}")

df_analyse.nlargest(6, "health_exp")[["country", "year", "health_exp", "gdp"]]

Zéros de dépenses de santé : 606
Zéros en 2015 : 181
Valeurs non nulles en 2015 : 2
Valeurs supérieures à 100 : 1301
Valeurs supérieures à 1 000 : 413
Ratio médian dépenses de santé / PIB : 0.096


,country,year,health_exp,gdp
2522,Switzerland,2014,19479.91161,85814.58857
2523,Switzerland,2013,19099.04506,84658.88768
1545,Luxembourg,2008,18961.34860,114293.84330
2525,Switzerland,2011,18822.86732,87998.44468
2524,Switzerland,2012,18379.32974,83164.38795
1542,Luxembourg,2011,17028.52798,115761.57700


Une dépense de santé exprimée en pourcentage ne peut pas dépasser 100 %. Or 1 301 lignes dépassent 100, dont 413 dépassent 1 000, avec un maximum à 19 480 pour la Suisse. Les 611 zéros (606 après retrait des 10 lignes sans espérance de vie) et les valeurs élevées chez les pays riches suggèrent plutôt une dépense par habitant en dollars qu'un pourcentage. Le ratio médian entre `health_exp` et le PIB est de 9,6 %, ce qui est l'ordre de grandeur d'une part de santé dans le PIB.

Je décide donc de la réintégrer comme proxy de dépense par habitant : les zéros deviennent des manquants, la variable est passée en log10, et les analyses sont limitées à 2000-2014. En 2015, seules 2 valeurs sont non nulles. Je la garde comme variable secondaire, car sa corrélation avec l'espérance de vie est largement expliquée par le PIB.

In [10]:
# Les zéros de scolarité sont-ils de vraies valeurs ou des données manquantes ?
exemples_scolarite = ["Antigua and Barbuda", "South Sudan", "Montenegro"]

scolarite_series = (
    df_analyse[df_analyse["country"].isin(exemples_scolarite)]
    .pivot(index="year", columns="country", values="schooling")
)
scolarite_series

country,Antigua and Barbuda,Montenegro,South Sudan
year,,,
2000,0.0,0.0,0.0
2001,0.0,0.0,0.0
2002,0.0,0.0,0.0
2003,0.0,0.0,0.0
2004,0.0,12.6,0.0
2005,0.0,12.8,0.0
2006,14.7,13.2,0.0
2007,14.5,13.6,0.0
2008,14.4,14.2,0.0


La scolarité à zéro n'est pas crédible comme valeur réelle. Antigua et Barbuda restent à 0 de 2000 à 2005, puis passent à 14,7 en 2006. Le Monténégro est à 0 jusqu'en 2003, puis à 12,6. Le Soudan du Sud est à 0 jusqu'en 2010, puis à 4,9.

Ces zéros ressemblent à des codes de données manquantes. Je les transforme donc en valeurs manquantes avant l'analyse. En revanche, je peux les interpoler pour l'affichage, car la scolarité évolue lentement et la série est continue autour des zéros.

In [11]:
# Je quantifie les valeurs extrêmes, mais je ne les supprime pas automatiquement :
# sur une distribution asymétrique, la règle des 1,5 IQR signale surtout la forme de la distribution.
def compter_extremes(serie, seuil=1.5):
    # Nombre de valeurs hors de l'intervalle [Q1 - 1,5×IQR ; Q3 + 1,5×IQR].
    serie = serie.dropna()
    q1, q3 = serie.quantile([0.25, 0.75])
    ecart = q3 - q1
    return int(((serie < q1 - seuil * ecart) | (serie > q3 + seuil * ecart)).sum())


variables_controle = [
    "life_expectancy", "adult_mortality", "infant_deaths", "alcohol", "measles",
    "under_five_deaths", "gdp", "income_comp", "schooling", "hiv_aids",
]

controle = pd.DataFrame(
    [
        (LABELS[c], compter_extremes(df_analyse[c]), df_analyse[c].notna().sum())
        for c in variables_controle
    ],
    columns=["Variable", "Valeurs hors 1,5 IQR", "Observations"],
)
controle["Part (%)"] = (controle["Valeurs hors 1,5 IQR"] / controle["Observations"] * 100).round(1)
controle.sort_values("Valeurs hors 1,5 IQR", ascending=False)

,Variable,"Valeurs hors 1,5 IQR",Observations,Part (%)
9,Décès VIH/sida (pour 1 000 naissances),542,2928,18.5
4,Cas de rougeole (pour 1 000),542,2928,18.5
5,Décès avant 5 ans (pour 1 000),394,2928,13.5
6,PIB par habitant (USD),363,2485,14.6
2,Décès infantiles (pour 1 000),315,2928,10.8
7,Indice de composition des revenus (0-1),130,2768,4.7
1,Mortalité adulte (pour 1 000),82,2928,2.8
8,Scolarité (années),42,2768,1.5
0,Espérance de vie (années),10,2928,0.3
3,Consommation d'alcool (L/hab.),0,2735,0.0


La règle des 1,5 IQR signale beaucoup de valeurs pour les variables très asymétriques, comme la rougeole, le VIH/sida ou la mortalité infantile. Ce n'est pas une preuve d'erreur : une épidémie ou une crise sanitaire produit réellement des valeurs extrêmes.

Je garde donc ces valeurs. Pour limiter leur influence, j'utiliserai les médianes et les échelles logarithmiques plutôt que les moyennes seules. Je ne supprime que les valeurs impossibles ou manifestement mal saisies.

## 4. Nettoyer et sauvegarder

Je résume les décisions prises et je les applique dans une copie du fichier. Je garde aussi des indicateurs de manquant pour les variables où le fait de manquer est susceptible d'être informatif.

In [12]:
# Je construis le jeu de données d'analyse, sans imputation.
df_propre = df_analyse.copy()
df_propre["statut"] = df_propre["status"].map(STATUTS)

# Les zéros de scolarité sont traités comme des données manquantes.
zeros_scolarite = (df_propre["schooling"] == 0).sum()
df_propre["schooling"] = df_propre["schooling"].mask(df_propre["schooling"] == 0)

# Je signale les IMC impossibles, sans utiliser cette variable dans les conclusions.
df_propre["bmi_suspect"] = df_propre["bmi"].lt(10) | df_propre["bmi"].gt(80)
df_propre["bmi"] = df_propre["bmi"].where(~df_propre["bmi_suspect"])

# Dépenses de santé : zéro = manquant, 2015 exclue, log10 pour la distribution.
df_propre["health_exp_zero"] = df_propre["health_exp"].eq(0)
df_propre["health_exp_disponible"] = (
    df_propre["year"].lt(2015) & df_propre["health_exp"].gt(0)
)
df_propre["health_exp_log"] = np.log10(
    df_propre["health_exp"].where(df_propre["health_exp_disponible"])
)

# PIB : je signale les valeurs aberrantes sans écraser la colonne brute.
df_propre = marquer_pib_aberrant(df_propre)

# Indicateurs de manquant : ils seront utiles pour vérifier si le manque est informatif.
for colonne in ["gdp", "hepatitis_b", "schooling", "income_comp"]:
    df_propre[f"{colonne}_manquant"] = df_propre[colonne].isna()

# Je ne touche pas à population et bmi pour l'analyse :
# ces colonnes sont soit incohérentes, soit d'échelle douteuse.
print(f"Lignes conservées : {len(df_propre):,}")
print(f"Zéros de scolarité remplacés par NaN : {zeros_scolarite}")
print(f"IMC signalés comme suspects : {int(df_propre['bmi_suspect'].sum())}")
print(f"Dépenses de santé exploitables (2000-2014) : {int(df_propre['health_exp_disponible'].sum())}")
print(f"PIB signalés comme aberrants : {int(df_propre['gdp_aberrant'].sum())}")
print(f"PIB exploitables après signalement : {int(df_propre['gdp_clean'].notna().sum())}")
print(f"Pays : {df_propre['country'].nunique()}")
print(f"Années par pays : {sorted(df_propre.groupby('country').size().unique().tolist())}")

df_propre.to_csv(DATA_OBS, index=False, encoding="utf-8")
print(f"\nFichier écrit : {DATA_OBS.relative_to(RACINE)}")

Lignes conservées : 2,928
Zéros de scolarité remplacés par NaN : 26
IMC signalés comme suspects : 282
Dépenses de santé exploitables (2000-2014) : 2320
PIB signalés comme aberrants : 622
PIB exploitables après signalement : 1863
Pays : 183
Années par pays : [16]

Fichier écrit : data\clean\life_expectancy_observations.csv


Le fichier `life_expectancy_observations.csv` contient les données nettoyées, sans imputation. Les valeurs manquantes restent visibles, et les colonnes Population, IMC et dépenses de santé en pourcentage restent dans le fichier pour pouvoir vérifier mes constats, mais elles ne seront pas utilisées dans les analyses.

In [13]:
# Pour le dashboard, je prépare une version séparée.
# Je n'impute que les variables pour lesquelles l'interpolation est défendable :
# scolarité, indice de revenu, polio et DTP3.
df_dashboard = df_propre.sort_values(["country", "year"]).reset_index(drop=True).copy()
variables_interpolees = ["schooling", "income_comp", "polio", "diphtheria"]

for variable in variables_interpolees:
    df_dashboard[f"{variable}_estimee"] = df_dashboard[variable].isna()
    # Interpolation dans la série du pays, puis médiane de l'année si le pays n'a aucune valeur.
    df_dashboard[variable] = df_dashboard.groupby("country")[variable].transform(
        lambda serie: serie.interpolate(limit_direction="both")
    )
    df_dashboard[variable] = df_dashboard.groupby("year")[variable].transform(
        lambda serie: serie.fillna(serie.median())
    )
    df_dashboard[variable] = df_dashboard[variable].fillna(df_dashboard[variable].median())

bilan_interpolation = pd.DataFrame({
    "Variable": [LABELS[v] for v in variables_interpolees],
    "Valeurs estimées": [int(df_dashboard[f"{v}_estimee"].sum()) for v in variables_interpolees],
})
print(bilan_interpolation.to_string(index=False))

# Je retire du fichier dashboard les colonnes que je ne juge pas exploitables.
# health_exp reste : elle est maintenant utilisable en log (2000-2014).
colonnes_exclues = ["population", "bmi", "bmi_suspect"]
df_dashboard = df_dashboard.drop(columns=colonnes_exclues)
df_dashboard.to_csv(DATA_DASH, index=False, encoding="utf-8")
print(f"\nFichier écrit : {DATA_DASH.relative_to(RACINE)}")
print(f"Colonnes retirées : {', '.join(LABELS[c] for c in colonnes_exclues)}")
print(f"Colonnes du dashboard : {df_dashboard.shape[1]}")

                               Variable  Valeurs estimées
                     Scolarité (années)               186
Indice de composition des revenus (0-1)               160
                  Vaccination polio (%)                19
                   Vaccination DTP3 (%)                19



Fichier écrit : data\clean\life_expectancy_dashboard.csv
Colonnes retirées : Population, IMC moyen, IMC signalé comme suspect
Colonnes du dashboard : 34


Pourquoi seulement ces quatre variables ? Parce que la scolarité, l'indice de revenu et les taux de vaccination polio/DTP3 manquent peu ou de façon continue dans le temps, et que leur interpolation dans la série d'un pays est réaliste. À l'inverse, je n'impute pas l'hépatite B, le PIB, l'alcool, les dépenses publiques ni les dépenses de santé : le manque y est trop lié à l'année ou au niveau de développement pour qu'une interpolation soit neutre. Les dépenses de santé restent observées et limitées à 2000-2014.

L'interpolation n'est pas une vérité. C'est un choix d'affichage, et les colonnes `_estimee` permettent de savoir quelles valeurs ont été calculées. Les analyses statistiques, elles, restent sur les valeurs observées.

### Est-ce que mes choix changent les conclusions ?

C'est la question que je dois me poser avant d'interpréter les corrélations. Je compare trois façons de traiter les manquants :

1. les corrélations calculées sur les paires de valeurs observées ;
2. les corrélations calculées sur les lignes complètes pour toutes les variables ;
3. les corrélations obtenues après une imputation simple par médiane du pays, puis de l'année, puis globale.

Si les trois donnent des résultats proches, mes conclusions sont robustes. Si elles changent beaucoup, je dois être beaucoup plus prudent.

In [14]:
variables_sensibilite = [
    "schooling", "income_comp", "adult_mortality", "hiv_aids",
    "gdp", "polio", "diphtheria",
]

# 1) Corrélations sur les paires observées.
# 2) Corrélations sur les lignes complètes pour toutes les variables.
complet = df_propre[["life_expectancy"] + variables_sensibilite].dropna()

# 3) Imputation simple, uniquement pour mesurer l'écart.
df_impute = df_propre.copy()
for variable in variables_sensibilite:
    df_impute[variable] = df_impute.groupby("country")[variable].transform(
        lambda serie: serie.fillna(serie.median())
    )
    df_impute[variable] = df_impute.groupby("year")[variable].transform(
        lambda serie: serie.fillna(serie.median())
    )
    df_impute[variable] = df_impute[variable].fillna(df_impute[variable].median())

lignes_sensibilite = []
for variable in variables_sensibilite:
    observe = df_propre[["life_expectancy", variable]].dropna()
    lignes_sensibilite.append({
        "Variable": LABELS[variable],
        "r observé": observe["life_expectancy"].corr(observe[variable]),
        "n observé": len(observe),
        "r cas complets": complet["life_expectancy"].corr(complet[variable]),
        "r après imputation": df_impute["life_expectancy"].corr(df_impute[variable]),
    })

sensibilite = pd.DataFrame(lignes_sensibilite)
print(f"Taille des cas complets : {len(complet):,} lignes")
sensibilite.round(3)

Taille des cas complets : 2,462 lignes


,Variable,r observé,n observé,r cas complets,r après imputation
0,Scolarité (années),0.784,2742,0.789,0.752
1,Indice de composition des revenus (0-1),0.725,2768,0.757,0.690
2,Mortalité adulte (pour 1 000),-0.696,2928,-0.686,-0.696
3,Décès VIH/sida (pour 1 000 naissances),-0.557,2928,-0.570,-0.557
4,PIB par habitant (USD),0.461,2485,0.462,0.434
5,Vaccination polio (%),0.466,2909,0.456,0.472
6,Vaccination DTP3 (%),0.479,2909,0.468,0.485


Les résultats sont globalement stables, mais pas identiques. Pour la scolarité, la corrélation passe de 0,784 sur les valeurs observées à 0,789 sur les cas complets, puis à 0,752 après imputation. Pour l'indice de revenu, elle passe de 0,725 à 0,757, puis à 0,690. Pour la mortalité adulte et le VIH/sida, elle ne bouge presque pas.

L'imputation simple a donc tendance à atténuer légèrement le lien entre l'éducation, le revenu, le PIB et l'espérance de vie. Les conclusions principales restent les mêmes, mais je préfère travailler sur les valeurs observées et signaler cette limite, plutôt que d'annoncer des chiffres calculés sur des données inventées.


### Dépenses de santé et PIB

Avant de réintégrer les dépenses de santé, je vérifie ce qu'elles apportent vraiment. Une corrélation brute peut venir du fait que les pays riches dépensent plus et vivent plus longtemps. Je calcule donc aussi une corrélation partielle, en contrôlant le PIB.


In [15]:

# Corrélations des dépenses de santé et du PIB avec l'espérance de vie.
df_sante = df_propre.copy()
df_sante["gdp_log"] = np.log10(df_sante["gdp"].where(df_sante["gdp"] > 0))
df_sante["gdp_clean_log"] = np.log10(df_sante["gdp_clean"].where(df_sante["gdp_clean"] > 0))

paires_sante = df_sante[["life_expectancy", "health_exp_log"]].dropna()
paires_gdp = df_sante[["life_expectancy", "gdp_log"]].dropna()
paires_gdp_clean = df_sante[["life_expectancy", "gdp_clean_log"]].dropna()

r_sante = paires_sante["life_expectancy"].corr(paires_sante["health_exp_log"])
r_gdp = paires_gdp["life_expectancy"].corr(paires_gdp["gdp_log"])
r_gdp_clean = paires_gdp_clean["life_expectancy"].corr(paires_gdp_clean["gdp_clean_log"])
r_partiel_brut, n_partiel_brut = correlation_partielle(
    df_sante["health_exp_log"], df_sante["life_expectancy"], df_sante["gdp_log"]
)
r_partiel_clean, n_partiel_clean = correlation_partielle(
    df_sante["health_exp_log"], df_sante["life_expectancy"], df_sante["gdp_clean_log"]
)

pd.DataFrame([
    {"Relation": "Dépenses santé (log) / espérance de vie", "r": r_sante, "n": len(paires_sante)},
    {"Relation": "PIB brut (log) / espérance de vie", "r": r_gdp, "n": len(paires_gdp)},
    {"Relation": "PIB nettoyé (log) / espérance de vie", "r": r_gdp_clean, "n": len(paires_gdp_clean)},
    {"Relation": "Dépenses santé, contrôle PIB brut", "r": r_partiel_brut, "n": n_partiel_brut},
    {"Relation": "Dépenses santé, contrôle PIB nettoyé", "r": r_partiel_clean, "n": n_partiel_clean},
]).round(3)


,Relation,r,n
0,Dépenses santé (log) / espérance de vie,0.597,2320
1,PIB brut (log) / espérance de vie,0.598,2485
2,PIB nettoyé (log) / espérance de vie,0.654,1863
3,"Dépenses santé, contrôle PIB brut",0.116,2320
4,"Dépenses santé, contrôle PIB nettoyé",0.072,1731



Les dépenses de santé sont fortement liées à l'espérance de vie en apparence, mais une fois le PIB contrôlé, le lien s'effondre. L'essentiel de l'association passe donc par le niveau de richesse. Je garde la variable comme indicateur secondaire pour la page Preston, pas comme levier indépendant.


## 5. Décrire les variables

Je décris maintenant les variables que je juge fiables. Je laisse de côté Population et IMC, pour les raisons vues plus haut. Pour le PIB, j'utilise la version nettoyée `gdp_clean` ; pour les dépenses de santé, la version logarithmique limitée à 2000-2014.

Certaines colonnes ont moins d'observations que les 2 928 lignes : c'est normal, les statistiques sont calculées sur les valeurs disponibles.

In [16]:
# Je décris uniquement les variables que je juge fiables.
# gdp_clean et health_exp_log ont moins d'observations : c'est attendu.
variables_description = [
    "life_expectancy", "adult_mortality", "infant_deaths", "under_five_deaths",
    "alcohol", "total_exp", "hepatitis_b", "measles", "polio", "diphtheria",
    "hiv_aids", "gdp_clean", "thinness_10_19", "thinness_5_9", "income_comp",
    "schooling", "health_exp_log",
]

stats = df_propre[variables_description].describe().T
stats.index = [LABELS[c] for c in stats.index]
stats = stats.rename(columns={
    "count": "N", "mean": "Moyenne", "std": "Écart-type", "min": "Min",
    "25%": "Q1", "50%": "Médiane", "75%": "Q3", "max": "Max",
})
stats.round(2)

,N,Moyenne,Écart-type,Min,Q1,Médiane,Q3,Max
Espérance de vie (années),2928.0,69.22,9.52,36.30,63.10,72.10,75.70,89.00
Mortalité adulte (pour 1 000),2928.0,164.80,124.29,1.00,74.00,144.00,228.00,723.00
Décès infantiles (pour 1 000),2928.0,30.41,118.11,0.00,0.00,3.00,22.00,1800.00
Décès avant 5 ans (pour 1 000),2928.0,42.18,160.70,0.00,0.00,4.00,28.00,2500.00
Consommation d'alcool (L/hab.),2735.0,4.61,4.05,0.01,0.90,3.77,7.72,17.87
Dépenses publiques de santé (% du budget),2702.0,5.93,2.48,0.37,4.26,5.75,7.49,17.60
Vaccination hépatite B (%),2375.0,80.96,25.02,1.00,77.00,92.00,97.00,99.00
Cas de rougeole (pour 1 000),2928.0,2427.86,11485.97,0.00,0.00,17.00,362.25,212183.00
Vaccination polio (%),2909.0,82.55,23.42,3.00,78.00,93.00,97.00,99.00
Vaccination DTP3 (%),2909.0,82.32,23.71,2.00,78.00,93.00,97.00,99.00


In [17]:
# Comparaison des deux groupes de pays.
par_statut = (
    df_propre.groupby("statut")["life_expectancy"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
par_statut.columns = ["N", "Moyenne", "Médiane", "Écart-type"]
par_statut

,N,Moyenne,Médiane,Écart-type
statut,,,,
Développé,512,79.20,79.25,3.93
En développement,2416,67.11,69.00,9.01


In [18]:
# Quelques chiffres de synthèse pour le dashboard.
derniere_annee = df_propre["year"].max()
monde_2015 = df_propre.loc[df_propre["year"] == derniere_annee, "life_expectancy"]
moyennes_statut = par_statut["Moyenne"].to_dict()
ecart_statuts = moyennes_statut["Développé"] - moyennes_statut["En développement"]

print(f"Espérance de vie en {derniere_annee} : {monde_2015.mean():.1f} ans en moyenne")
print(f"Médiane : {monde_2015.median():.1f} ans")
print(f"Minimum : {monde_2015.min():.1f} ans, maximum : {monde_2015.max():.1f} ans")
print(f"Écart moyen développés / en développement, toutes années : {ecart_statuts:.1f} ans")

Espérance de vie en 2015 : 71.6 ans en moyenne
Médiane : 73.9 ans
Minimum : 51.0 ans, maximum : 88.0 ans
Écart moyen développés / en développement, toutes années : 12.1 ans


Sur toute la période, l'espérance de vie moyenne est de 69,2 ans, mais la médiane est plus élevée, à 72,1 ans. Cela indique une distribution tirée vers le bas par un groupe de pays à faible espérance de vie.

L'écart entre les deux groupes est important : 79,2 ans en moyenne pour les pays développés, contre 67,1 ans pour les pays en développement, soit 12,1 ans. En 2015, la moyenne mondiale atteint 71,6 ans et l'écart entre statuts est de 11,0 ans.

## 6. Regarder chaque variable

Je commence par un panorama des variables retenues, puis je détaille l'espérance de vie. L'objectif est de voir la forme des distributions avant de chercher des relations entre variables.

In [19]:
# Panorama des distributions : un sous-graphique par variable.
# make_subplots crée des axes indépendants ; go.Histogram n'utilise pas bingroup.
variables_panorama = [
    "life_expectancy", "adult_mortality", "infant_deaths", "under_five_deaths",
    "alcohol", "total_exp", "hepatitis_b", "measles", "polio", "diphtheria",
    "hiv_aids", "gdp_clean", "thinness_10_19", "thinness_5_9",
    "income_comp", "schooling", "health_exp_log",
]

df_hist = df_propre[variables_panorama].copy()

# Le PIB et la rougeole sont très étalés : je les passe en logarithme.
# health_exp_log est déjà en logarithme.
for colonne in ["gdp_clean", "measles"]:
    df_hist[colonne] = np.log10(df_hist[colonne].where(df_hist[colonne] > 0))

libelles_courts = {
    "life_expectancy": "Espérance de vie (ans)",
    "adult_mortality": "Mortalité adulte",
    "infant_deaths": "Décès infantiles",
    "under_five_deaths": "Décès avant 5 ans",
    "alcohol": "Alcool",
    "total_exp": "Dépenses publiques",
    "hepatitis_b": "Hépatite B",
    "measles": "Rougeole (log)",
    "polio": "Polio",
    "diphtheria": "DTP3",
    "hiv_aids": "VIH/sida",
    "gdp_clean": "PIB nettoyé (log)",
    "thinness_10_19": "Maigreur 10-19",
    "thinness_5_9": "Maigreur 5-9",
    "income_comp": "Indice de revenu",
    "schooling": "Scolarité",
    "health_exp_log": "Dépenses santé (log)",
}

NB_BINS = {
    "life_expectancy": 30, "adult_mortality": 30, "infant_deaths": 30,
    "under_five_deaths": 30, "alcohol": 25, "total_exp": 25,
    "hepatitis_b": 20, "measles": 25, "polio": 20, "diphtheria": 20,
    "hiv_aids": 30, "gdp_clean": 30, "thinness_10_19": 25,
    "thinness_5_9": 25, "income_comp": 25, "schooling": 25,
    "health_exp_log": 30,
}

NB_COLS = 4
NB_LIGNES = int(np.ceil(len(variables_panorama) / NB_COLS))

fig = make_subplots(
    rows=NB_LIGNES,
    cols=NB_COLS,
    subplot_titles=[libelles_courts[v] for v in variables_panorama],
    horizontal_spacing=0.06,
    vertical_spacing=0.07,
)

for i, variable in enumerate(variables_panorama):
    ligne = i // NB_COLS + 1
    colonne = i % NB_COLS + 1
    fig.add_trace(
        go.Histogram(
            x=df_hist[variable].dropna(),
            nbinsx=NB_BINS.get(variable, 25),
            marker_color=BLEU,
            name=libelles_courts[variable],
        ),
        row=ligne,
        col=colonne,
    )

fig.update_xaxes(matches=None, showticklabels=True)
fig.update_yaxes(matches=None, rangemode="tozero")

# Les 3 emplacements vides de la dernière ligne ne doivent pas apparaître.
for colonne in range(2, NB_COLS + 1):
    fig.update_xaxes(visible=False, row=NB_LIGNES, col=colonne)
    fig.update_yaxes(visible=False, row=NB_LIGNES, col=colonne)

fig.update_layout(
    title="Distribution des variables retenues",
    template="plotly_white",
    height=1400,
    showlegend=False,
    bargap=0.05,
)
fig.update_xaxes(tickfont=dict(size=9))
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

Les formes sont très différentes, et aucune n'est une cloche parfaitement symétrique.

L'espérance de vie est étirée vers la gauche : elle a un mode principal autour de 73 ans, un mode secondaire vers 57 ans et une queue de pays à faible espérance de vie. La moyenne (69,2 ans) est inférieure à la médiane (72,1 ans), ce qui confirme le poids de ces pays.

Les décès infantiles, les décès avant 5 ans, le VIH/sida et la rougeole sont massivement concentrés près de zéro, avec une longue queue de valeurs élevées. Les vaccinations polio, DTP3 et hépatite B sont au contraire massées près de 100.

Le PIB et les dépenses de santé sont beaucoup plus lisibles en logarithme. La scolarité et l'indice de revenu ont une forme plus régulière, sans être des cloches parfaites. Il faut donc adapter l'indicateur de tendance à chaque variable : la médiane est souvent plus parlante que la moyenne.

In [20]:
# Moyenne et médiane pour montrer l'asymétrie de la distribution.
moyenne = df_propre["life_expectancy"].mean()
mediane = df_propre["life_expectancy"].median()

fig = px.histogram(
    df_propre,
    x="life_expectancy",
    nbins=30,
    color_discrete_sequence=[BLEU],
    labels={"life_expectancy": "Espérance de vie (années)"},
    title="Distribution de l'espérance de vie",
)
fig.add_vline(
    x=moyenne, line_dash="dash", line_color=ORANGE,
    annotation_text=f"Moyenne : {moyenne:.1f} ans", annotation_position="top",
)
fig.add_vline(
    x=mediane, line_dash="dot", line_color=VERT,
    annotation_text=f"Médiane : {mediane:.1f} ans", annotation_position="bottom",
)
fig.update_layout(template="plotly_white", yaxis_title="Nombre de pays-années")
fig.show()

La distribution va de 36,3 à 89,0 ans. Elle n'est pas symétrique : elle est étirée vers la gauche, avec un mode principal autour de 73 ans, un mode secondaire vers 57 ans et une queue de pays à faible espérance de vie.

La moyenne (69,2 ans) est inférieure à la médiane (72,1 ans), ce qui confirme le poids des pays à faible espérance de vie. Environ un quart des observations se situe sous 63,1 ans.

In [21]:
# Boîte à moustaches pour comparer les deux statuts.
fig = px.box(
    df_propre,
    x="statut",
    y="life_expectancy",
    color="statut",
    color_discrete_map=COULEURS_STATUT,
    points="outliers",
    labels={"statut": "", "life_expectancy": "Espérance de vie (années)"},
    title="Espérance de vie selon le statut du pays",
)
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

La médiane des pays développés est autour de 79 ans, contre environ 69 ans pour les pays en développement. L'écart médian est donc proche de 10 ans.

Les deux groupes ont des valeurs atypiques, mais elles ne racontent pas la même chose : quelques pays développés ont des épisodes sanitaires dégradés, tandis que certains pays en développement cumulent les difficultés. Le statut ONU sera donc un axe utile pour le dashboard.

In [22]:
# Moyenne par année et par statut.
evolution = df_propre.groupby(["year", "statut"], as_index=False)["life_expectancy"].mean()
moy_2000 = df_propre.loc[df_propre["year"] == 2000, "life_expectancy"].mean()
moy_2015 = df_propre.loc[df_propre["year"] == 2015, "life_expectancy"].mean()

fig = px.line(
    evolution,
    x="year",
    y="life_expectancy",
    color="statut",
    color_discrete_map=COULEURS_STATUT,
    markers=True,
    labels={
        "year": "Année",
        "life_expectancy": "Espérance de vie moyenne (années)",
        "statut": "Statut",
    },
    title="Évolution de l'espérance de vie entre 2000 et 2015",
)
fig.add_annotation(
    x=0.02, y=0.04, xref="paper", yref="paper",
    text=f"+{moy_2015 - moy_2000:.1f} ans en moyenne mondiale",
    showarrow=False,
    align="left",
    bgcolor="white",
    bordercolor=GRIS,
)
fig.update_layout(template="plotly_white")
fig.show()

L'espérance de vie mondiale moyenne passe de 66,8 ans en 2000 à 71,6 ans en 2015, soit +4,9 ans en quinze ans.

Les pays en développement progressent plus vite (64,6 à 69,7 ans, +5,1 ans) que les pays développés (76,8 à 80,7 ans, +3,9 ans). L'écart se réduit donc légèrement, de 12,2 à 11,0 ans.

In [23]:
# Classement des pays en 2015 : 10 meilleurs et 10 moins bons.
derniere = df_propre[df_propre["year"] == derniere_annee]
hauts = derniere.nlargest(10, "life_expectancy")[["country", "life_expectancy"]].assign(groupe="Top 10")
bas = derniere.nsmallest(10, "life_expectancy")[["country", "life_expectancy"]].assign(groupe="Flop 10")
classement = pd.concat([hauts, bas]).sort_values("life_expectancy")

fig = px.bar(
    classement,
    x="life_expectancy",
    y="country",
    orientation="h",
    color="groupe",
    color_discrete_map={"Top 10": VERT, "Flop 10": ORANGE},
    labels={"life_expectancy": "Espérance de vie (années)", "country": "", "groupe": ""},
    title=f"Les 10 meilleurs et les 10 moins bons pays en {derniere_annee}",
)
fig.update_layout(template="plotly_white", height=650)
fig.show()

En 2015, l'écart entre la Slovénie, à 88,0 ans, et la Sierra Leone, à 51,0 ans, atteint 37 ans.

Le bas du classement est concentré en Afrique subsaharienne. Cela ne prouve pas que le statut ou la région soient des causes directes, mais cela indique où les situations sanitaires sont les plus dégradées.

In [24]:
# Carte mondiale : je convertis les noms de pays en codes ISO3.
MANUEL_ISO3 = {
    "Bolivia (Plurinational State of)": "BOL",
    "Côte d'Ivoire": "CIV",
    "Democratic People's Republic of Korea": "PRK",
    "Democratic Republic of the Congo": "COD",
    "Iran (Islamic Republic of)": "IRN",
    "Lao People's Democratic Republic": "LAO",
    "Micronesia (Federated States of)": "FSM",
    "Republic of Korea": "KOR",
    "Republic of Moldova": "MDA",
    "Russian Federation": "RUS",
    "Swaziland": "SWZ",
    "Syrian Arab Republic": "SYR",
    "The former Yugoslav republic of Macedonia": "MKD",
    "Timor-Leste": "TLS",
    "Turkey": "TUR",
    "United Kingdom of Great Britain and Northern Ireland": "GBR",
    "United Republic of Tanzania": "TZA",
    "United States of America": "USA",
    "Venezuela (Bolivarian Republic of)": "VEN",
    "Viet Nam": "VNM",
}


def vers_iso3(nom):
    # Quelques noms du fichier ne sont pas reconnus directement par pycountry.
    if nom in MANUEL_ISO3:
        return MANUEL_ISO3[nom]
    try:
        return pycountry.countries.lookup(nom).alpha_3
    except LookupError:
        return None


carte = derniere.copy()
carte["iso3"] = carte["country"].map(vers_iso3)
non_mappes = carte.loc[carte["iso3"].isna(), "country"].tolist()
print("Pays sans code ISO :", non_mappes if non_mappes else "aucun")

fig = px.choropleth(
    carte,
    locations="iso3",
    color="life_expectancy",
    color_continuous_scale="RdYlGn",
    range_color=(50, 85),
    labels={"life_expectancy": "Espérance de vie (ans)"},
    title=f"Espérance de vie dans le monde en {derniere_annee}",
)
fig.update_layout(
    template="plotly_white",
    coloraxis_colorbar_title="Années",
    margin=dict(l=0, r=0, t=60, b=0),
    geo=dict(showframe=False, showcoastlines=False),
)
fig.show()

Pays sans code ISO : aucun


La carte montre un gradient géographique net. Les valeurs élevées se trouvent surtout en Europe de l'Ouest, au Japon, en Australie, au Canada et aux États-Unis. Les valeurs les plus faibles concernent l'Afrique subsaharienne et quelques pays en crise.

Cette carte est un bon point de départ pour le dashboard : elle donne une lecture immédiate, sans nécessiter de connaître les variables.

## 7. Croiser les variables

Je cherche maintenant les variables qui varient avec l'espérance de vie. Je garde uniquement celles que je considère comme fiables, et je rappelle que deux variables qui évoluent ensemble ne sont pas forcément liées par une relation de cause à effet.

In [25]:
# Corrélations avec l'espérance de vie, triées pour faire ressortir les liens.
# health_exp_log n'est disponible que sur 2000-2014.
variables_corr = [
    "adult_mortality", "infant_deaths", "under_five_deaths", "measles",
    "hepatitis_b", "polio", "diphtheria", "hiv_aids", "alcohol", "total_exp",
    "gdp_clean", "income_comp", "schooling", "thinness_10_19", "thinness_5_9",
    "health_exp_log",
]

corr_cible = (
    df_propre[variables_corr + ["life_expectancy"]]
    .corr()["life_expectancy"]
    .drop("life_expectancy")
    .sort_values()
)
corr_cible.index = [LABELS[c] for c in corr_cible.index]
couleurs = [VERT if valeur > 0 else ORANGE for valeur in corr_cible.values]

fig = px.bar(
    x=corr_cible.values,
    y=corr_cible.index,
    orientation="h",
    labels={"x": "Corrélation avec l'espérance de vie (r de Pearson)", "y": ""},
    title="Les variables les plus liées à l'espérance de vie",
)
fig.update_traces(marker_color=couleurs)
fig.add_vline(x=0, line_color=GRIS)
fig.update_layout(template="plotly_white", height=760)
fig.show()

Les liens les plus forts sont positifs avec la scolarité (r = 0,78) et l'indice de composition des revenus (r = 0,73), et négatifs avec la mortalité adulte (r = -0,70) et les décès liés au VIH/sida (r = -0,56).

La vaccination contre la diphtérie et la polio est associée positivement (environ 0,47-0,48), et la maigreur des 10-19 ans négativement (environ -0,47). Le PIB nettoyé est associé positivement (r = 0,65), plus fortement que le PIB brut (0,60). La consommation d'alcool et les dépenses publiques de santé ont des liens plus faibles.

Les dépenses de santé ont une corrélation brute d'environ 0,60, mais ce chiffre est trompeur : après contrôle du PIB, la corrélation partielle tombe à 0,12 avec le PIB brut et 0,07 avec le PIB nettoyé. L'essentiel de l'association passe donc par le niveau de richesse. Je garde cette variable comme indicateur secondaire pour la page Preston, pas comme levier indépendant.

J'ai volontairement retiré l'IMC et la population : leurs valeurs ne sont pas fiables, et une corrélation calculée dessus serait trompeuse.

In [26]:
# Matrice de corrélation pour repérer les variables redondantes.
variables_heatmap = [
    "life_expectancy", "adult_mortality", "hiv_aids", "thinness_10_19",
    "polio", "diphtheria", "gdp_clean", "income_comp", "schooling",
    "total_exp", "alcohol", "health_exp_log",
]
corr = df_propre[variables_heatmap].corr()
corr.index = [LABELS[c] for c in corr.index]
corr.columns = [LABELS[c] for c in corr.columns]

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Matrice de corrélation des variables retenues",
    aspect="auto",
)
fig.update_layout(template="plotly_white", height=820, xaxis_title="", yaxis_title="")
fig.update_xaxes(tickangle=45)
fig.show()

La matrice montre que les variables explicatives sont elles-mêmes corrélées. La scolarité, l'indice de revenu, la vaccination et les faibles taux de mortalité vont souvent de pair. Cela suggère des systèmes sanitaires et socio-économiques qui se renforcent mutuellement, plutôt qu'un facteur isolé.

Les dépenses de santé sont fortement corrélées au PIB, ce qui confirme qu'elles ne mesurent pas un levier indépendant. Certaines corrélations sont aussi redondantes : scolarité et indice de revenu, polio et DTP3. Pour le dashboard, il faudra donc choisir quelques indicateurs représentatifs plutôt que tout afficher.

In [27]:
# Nuage de points PIB nettoyé / espérance de vie, avec une tendance sur le log du PIB.
donnees = df_propre.dropna(subset=["gdp_clean", "life_expectancy"]).copy()
x_log = np.log10(donnees["gdp_clean"])
y = donnees["life_expectancy"]
coef = np.polyfit(x_log, y, 1)
r_log = np.corrcoef(x_log, y)[0, 1]
grille = np.linspace(x_log.min(), x_log.max(), 100)

fig = px.scatter(
    donnees,
    x="gdp_clean",
    y="life_expectancy",
    color="statut",
    color_discrete_map=COULEURS_STATUT,
    opacity=0.45,
    log_x=True,
    labels={
        "gdp_clean": "PIB par habitant nettoyé (USD, échelle log)",
        "life_expectancy": "Espérance de vie (années)",
        "statut": "Statut",
    },
    title="PIB nettoyé et espérance de vie : un lien réel mais non linéaire",
)
fig.add_trace(go.Scatter(
    x=10 ** grille,
    y=np.polyval(coef, grille),
    mode="lines",
    name=f"Tendance (r = {r_log:.2f})",
    line=dict(color=GRIS, dash="dash", width=2),
))
fig.update_layout(template="plotly_white", legend_title="")
fig.show()

Sur l'échelle logarithmique, le PIB nettoyé est corrélé à l'espérance de vie avec r ≈ 0,65, contre 0,60 avec le PIB brut. Le nettoyage renforce donc légèrement le lien, en retirant les valeurs qui correspondent manifestement à des erreurs d'échelle.

La relation n'est pas linéaire : les gains sont importants aux faibles revenus, puis ils diminuent quand le PIB augmente. À niveau de revenu comparable, les pays développés gardent souvent un avantage, ce qui montre que le revenu n'explique pas tout.

In [28]:
# Trois variables choisies pour illustrer des liens de nature différente.
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        f"Scolarité (r = {df_propre['schooling'].corr(df_propre['life_expectancy']):.2f})",
        f"Mortalité adulte (r = {df_propre['adult_mortality'].corr(df_propre['life_expectancy']):.2f})",
        f"VIH/sida (r = {df_propre['hiv_aids'].corr(df_propre['life_expectancy']):.2f})",
    ],
)

for i, variable in enumerate(["schooling", "adult_mortality", "hiv_aids"], start=1):
    for statut, couleur in COULEURS_STATUT.items():
        sous_ensemble = df_propre[df_propre["statut"] == statut]
        fig.add_trace(
            go.Scatter(
                x=sous_ensemble[variable],
                y=sous_ensemble["life_expectancy"],
                mode="markers",
                marker=dict(size=4, color=couleur, opacity=0.4),
                name=statut,
                showlegend=(i == 1),
            ),
            row=1,
            col=i,
        )

fig.update_xaxes(title_text="Années de scolarité", row=1, col=1)
fig.update_xaxes(title_text="Décès pour 1 000 adultes", row=1, col=2)
fig.update_xaxes(title_text="Décès pour 1 000 naissances", row=1, col=3)
fig.update_yaxes(title_text="Espérance de vie (années)", row=1, col=1)
fig.update_layout(
    template="plotly_white",
    title="Trois liens à regarder de près : scolarité, mortalité adulte et VIH/sida",
    height=460,
    legend_title="",
)
fig.show()

La scolarité est le lien positif le plus net. Le nuage est relativement resserré, même si l'écart entre statuts reste visible.

La mortalité adulte est le lien négatif le plus net. Elle résume en partie la qualité du système de santé et les conditions de vie.

Le VIH/sida est particulier : la plupart des pays sont proches de zéro, mais quelques pays d'Afrique australe et de l'Est cumulent des taux très élevés et une espérance de vie réduite. Cette concentration est importante pour le storytelling du dashboard.

## 8. Regarder l'évolution

Les moyennes mondiales cachent des trajectoires très différentes. Je calcule la variation de l'espérance de vie entre 2000 et 2015 pour chaque pays.

In [29]:
# Variation de l'espérance de vie entre 2000 et 2015, pays par pays.
pivot = df_propre.pivot(index="country", columns="year", values="life_expectancy")
progression = (pivot[2015] - pivot[2000]).dropna().sort_values()
gains = pd.concat([progression.tail(10), progression.head(5)]).sort_values()

gains_df = gains.reset_index()
gains_df.columns = ["country", "variation"]
gains_df["groupe"] = np.where(
    gains_df["variation"] > 0,
    "Plus fortes progressions",
    "Plus fortes baisses",
)

fig = px.bar(
    gains_df,
    x="variation",
    y="country",
    orientation="h",
    color="groupe",
    color_discrete_map={"Plus fortes progressions": VERT, "Plus fortes baisses": ORANGE},
    labels={
        "variation": "Variation de l'espérance de vie (années)",
        "country": "",
        "groupe": "",
    },
    title="Qui a le plus gagné et perdu entre 2000 et 2015 ?",
)
fig.add_vline(x=0, line_color=GRIS)
fig.update_layout(template="plotly_white", height=620)
fig.show()

Le Zimbabwe gagne 21,0 ans entre 2000 et 2015. L'Érythrée (+19,4), la Zambie (+18,0), le Botswana (+17,9) et le Rwanda (+17,8) suivent. Ces progressions sont cohérentes avec le recul de la mortalité liée au VIH/sida et l'amélioration de l'accès aux soins dans ces pays, mais je ne peux pas le démontrer avec ce seul jeu de données.

À l'opposé, la Syrie perd 8,1 ans, la Libye 5,3 ans et le Paraguay 5,0 ans. Les deux premiers cas suggèrent fortement l'effet des crises politiques et des conflits, mais là encore, c'est une interprétation à confirmer par d'autres sources.

In [30]:
# Quelques trajectoires nationales pour illustrer des évolutions différentes.
exemples = ["Zimbabwe", "Rwanda", "Japan", "Syrian Arab Republic"]
fig = px.line(
    df_propre[df_propre["country"].isin(exemples)],
    x="year",
    y="life_expectancy",
    color="country",
    markers=True,
    labels={"year": "Année", "life_expectancy": "Espérance de vie (années)", "country": "Pays"},
    title="Quatre trajectoires très différentes",
)
fig.update_layout(template="plotly_white", legend_title="")
fig.show()

Le Zimbabwe montre une chute puis un rebond. Le Rwanda montre une reconstruction rapide après le milieu des années 1990. Le Japon, déjà en haut de l'échelle, progresse lentement. La Syrie chute à partir de 2011.

Pour le dashboard, il faudra donc montrer à la fois le niveau et la trajectoire. Une carte ou un classement sur une seule année ne suffit pas à raconter ces histoires.

## 9. Ce que je retiens

### Les résultats

1. L'espérance de vie progresse de 4,9 ans en moyenne mondiale entre 2000 et 2015, de 66,8 à 71,6 ans.
2. Les inégalités restent fortes : 11 ans d'écart entre pays développés et pays en développement en 2015, et 37 ans entre le meilleur et le moins bon pays.
3. La scolarité est la variable la plus fortement associée à l'espérance de vie (r = 0,78), devant l'indice de revenu (0,73), la mortalité adulte (-0,70) et le VIH/sida (-0,56).
4. Le PIB nettoyé est lié à l'espérance de vie, mais de façon non linéaire : r ≈ 0,65 sur une échelle logarithmique, avec des gains qui diminuent quand le revenu augmente.
5. Les dépenses de santé sont corrélées à l'espérance de vie (r ≈ 0,60), mais leur corrélation partielle tombe à 0,12 avec le PIB brut et 0,07 avec le PIB nettoyé. C'est un indicateur secondaire, pas un levier indépendant.
6. Le VIH/sida pèse très lourdement dans quelques pays, mais ces pays connaissent aussi certaines des plus fortes progressions récentes.
7. Les trajectoires nationales sont très différentes : le Zimbabwe gagne 21 ans, la Syrie en perd 8 entre 2000 et 2015.

### Les limites

- Les données sont observationnelles. Une corrélation n'implique pas une causalité.
- L'analyse porte sur des moyennes nationales, pas sur des individus. On parle d'analyse écologique.
- La classification Développé / En développement provient de la source et surprend parfois : la France et le Canada y sont classés « En développement ». Je la conserve telle quelle pour ne pas réinterpréter les données.
- La colonne Population est incohérente et inutilisable : 0/143 séries cohérentes, 703 transitions > ×10, 129 > ×100. Je l'ai exclue.
- Le PIB contient de nombreuses valeurs aberrantes (plus de 600 observations signalées, 25 pays sans aucun PIB dans le fichier nettoyé). Je distingue `gdp` brut et `gdp_clean`, et j'exclus les valeurs signalées des tendances.
- La colonne IMC a une échelle douteuse. Elle est exclue des conclusions.
- Les dépenses de santé sont un proxy de dépense par habitant, pas un pourcentage. Les zéros sont traités comme des manquants, 2015 est exclue, et la corrélation est largement expliquée par le PIB.
- Les valeurs manquantes ne sont pas réparties au hasard. Pour l'hépatite B et la scolarité, le fait de manquer est lié à une espérance de vie plus faible. Une imputation simple peut donc biaiser les résultats.
- L'imputation change légèrement les corrélations : par exemple, scolarité et espérance de vie passent de 0,784 à 0,752. Les conclusions principales restent stables, mais les chiffres doivent être présentés avec cette réserve.
- Les dépenses publiques de santé, l'alcool et les dépenses de santé sont presque absents en 2015. Les analyses qui les utilisent doivent s'arrêter à 2014.
- La moyenne simple donne le même poids à tous les pays, quelle que soit leur population. Sans une colonne Population fiable, je ne peux pas calculer une moyenne pondérée.
- Les données s'arrêtent en 2015. Les conclusions doivent être datées.

### Ce que je mettrais en avant dans le dashboard

| KPI | Définition | Intérêt | Visualisation |
|---|---|---|---|
| Espérance de vie mondiale | Moyenne des pays, dernière année disponible | Niveau global | Carte KPI avec variation |
| Progression depuis 2000 | Écart de moyenne mondiale entre 2000 et 2015 | Tendance de fond | Courbe et KPI |
| Écart Nord/Sud | Différence de moyenne entre statuts | Inégalités | Barres groupées |
| Top et flop 5 pays | Meilleures et moins bonnes espérances de vie | Extremes | Barres horizontales |
| Classement des facteurs | Corrélations avec l'espérance de vie | Compréhension des liens | Barres horizontales |
| Carte mondiale | Espérance de vie par pays, dernière année | Lecture géographique | Choroplèthe |
| Évolution temporelle | Moyenne par statut, 2000-2015 | Dynamique | Courbes multiples |
| Courbe de Preston | PIB par habitant et espérance de vie, par pays et par année | Comparaison interactive | Petits multiples avec curseur temporel |
| Couverture des données | Part de pays renseignés pour un indicateur | Qualité de l'information | Jauge ou barre |

### Structure de dashboard que je propose

1. Vue d'ensemble : KPI mondiaux, carte, tendance 2000-2015.
2. Courbe de Preston : sélection de pays, curseur temporel, paramètres en petits multiples, tableau et synthèse.
3. Santé : mortalité adulte et infantile, VIH/sida, vaccination, rougeole.
4. Socio-économie : PIB, scolarité, indice de revenu, dépenses publiques de santé.
5. Comparateur de pays : filtres par pays et par année, courbes et indicateurs détaillés.

Je garderai les mêmes règles de lecture : un graphique pour un message, des titres explicites, des unités visibles, et une phrase de lecture sous chaque visualisation.

## 10. Reproductibilité

Pour relancer le notebook :

```bash
source .venv/Scripts/activate
jupyter lab
```

Les fichiers nettoyés sont dans `data/clean/` :

- `life_expectancy_observations.csv` : données nettoyées, sans imputation, avec `health_exp_log`, `gdp_clean` et `gdp_aberrant` ;
- `life_expectancy_dashboard.csv` : version d'affichage, avec interpolation de quatre variables seulement, les colonnes `_estimee` et les colonnes dérivées.

L'application Dash se lance avec `python app.py`. Elle utilise le fichier dashboard et propose les pages Vue d'ensemble, Courbe de Preston et Méthodologie.

In [31]:
# Versions utilisées pour reproduire l'analyse.
import sys

import plotly

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("numpy :", np.__version__)
print("plotly :", plotly.__version__)

Python : 3.13.5
pandas : 3.1.0
numpy : 2.5.4
plotly : 7.1.0
